# Jupiter to automize clustering for Cats

In [2]:
import pandas as pd
import numpy as np
import sklearn.metrics
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import autosklearn.classification

## I. Open the datasets

In [ ]:
df = pd.read_csv('../data/OutCatdata.csv', na_filter= False)
df = df.drop("Unnamed: 0", axis= 1)

/!\\ the NA values are dropped /!\\

## first look at the data

In [ ]:
df

In [ ]:
dfQuanti

### Split the datasets between classes to predict and data

In [ ]:
variables = df.drop(['Hunt'], axis = 1)
variables

In [ ]:
classes = df['Hunt']
classes

We have 2 resulting df : 

* classes consisting of the true Hunt status
* variables consisting of the factors

## Tentative de coller le TP bêtement

In [ ]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=120, 
    per_run_time_limit=20, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

*les param minimum pour la tâche* : 

- time_left_for_this_task= 2000 s
- per_run_time_limit=30 cycles
- n_jobs = 16 coeur
- memory_limit = 24 Go

### Génération des jeux de test, validation

pour notre entrainement, nous prenons des proportions de 67% de test et 33% de test

Les méthodes testées sont : 

* Forêt aléatoire
* Latent Dirichlet Allocation
* Multilayered Perceptron
* Baisien naif
* k plus proches voisins 

### Dans un premier temps, nous allons utiliser seulement les données Quantitatives

#### gestion de la suppression des colonnes quantitatives

In [ ]:
dfQuali = variables.drop(["event.id","timestamp","location.long","location.lat",
                          "animal.id","StartDate","StartHours","EndDate","EndHours"],
                         axis = 1).astype('category')

Crée le jeu de test qualitatif

In [ ]:
variables_trainQ, variables_testQ, classes_trainQ, classes_testQ = train_test_split(
        																dfQuali, classes, test_size = 0.33, random_state=0)

Transformation de classesQ en `category` à la place de `object`

In [ ]:
classes_testQ = classes_testQ.astype("category")
classes_trainQ = classes_trainQ.astype("category")

application des paramêtre afin de crée le modèle

In [ ]:
cls.fit(variables_trainQ, classes_trainQ)

Pour l'instant, pn voit que le modèle est claqué :(

	pire que tout, il fonctionne pas 

In [ ]:
cls.leaderboard()

In [ ]:
predictions = list(cls.predict(variables_testQ))

précision : 

In [ ]:
print("Accuracy score:", sklearn.metrics.accuracy_score(np.array(classes_testQ), predictions))

table des stats

In [ ]:
print( sklearn.metrics.classification_report(classes_testQ, predictions) )

décevant : on a un précision catastrophique.

Nous allons regarder la matrice de confusion pour potentiellement observer quel groupe est le mieu prédit

In [ ]:
np.round( confusion_matrix(classes_testQ, predictions), 3)

Le problème est clair : on prédit tout en une classe

## Tentative avec du quantitatif car le quali il fonctionne pas encore

### Ouverture du csv

In [3]:
dfQuanti = pd.read_csv('../data/OutCatdataQuantiNormZ.csv', na_filter= False)
dfQuanti = dfQuanti.drop("Unnamed: 0", axis= 1)

/home/jan_codage/miniforge3/envs/fouille/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3552: DtypeWarning: Columns (6,7,8) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


### Suppression des lignes ayant des Na

In [4]:
dfQuanti = dfQuanti[~np.any(dfQuanti == "NA",axis=1)]

### Création des sous jeux de données de test et d'entrainement

In [8]:
x = dfQuanti.drop('Hunt', axis=1).to_numpy().astype(np.float64)
y = dfQuanti.Hunt.astype(object)

y[y ==	-2.16472428] = "Na"
y[y == 	-0.78922734] = "No"
y[y ==  	0.5862696]  = "Yes"

x_train, x_test, y_train, y_test = train_test_split(x, y,
                                                    test_size = 0.33, random_state=0)

### Établissement d'un modèle

In [ ]:
cls = autosklearn.classification.AutoSklearnClassifier(
    time_left_for_this_task=120, 
    per_run_time_limit=20, 
    n_jobs=16,
    include = {"classifier":["random_forest",
                                "lda",
                                "mlp", # multilayer perceptron
                                "gaussian_nb", #  Naive Bayes
                                "k_nearest_neighbors"]},
    memory_limit=24576)

Ajustment du modèle a notre jeu de données

In [ ]:
cls.fit(x_train, y_train, dataset_name='Cat Data')

ValueError: Classification with data of type continuous is not supported. Supported types are ['binary', 'multiclass', 'multilabel-indicator']. You can find more information about scikit-learn data types in: https://scikit-learn.org/stable/modules/multiclass.html